In [ ]:
%pip install pyspark --quiet

In [ ]:
# Verifier que Java est disponible avant de lancer Spark.
import subprocess
try:
    r = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print((r.stderr or r.stdout).strip())
    print("\nJava detecte : on peut lancer Spark.")
except FileNotFoundError:
    print("Java INTROUVABLE.")
    print("Installer Temurin JDK 17 (https://adoptium.net),")
    print("puis redemarrer le noyau Jupyter avant de continuer.")

In [ ]:
from pathlib import Path
import sys, time
import pandas as pd

sys.path.append(r"E:\Projet_Alzheimer\configs")
from config_projet import C

from pyspark.sql import SparkSession, functions as F

# Session locale : local[*] = tous les coeurs de la machine.
spark = (SparkSession.builder
         .appName("ADNI_cohorte")
         .master("local[*]")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version, "| coeurs :", spark.sparkContext.defaultParallelism)

In [ ]:
# --- Chargement des tables en Spark ---
chemin_dx = str(C.table("All_Subjects_DXSUM_"))
chemin_irm = str(C.table("All_Subjects_Key_MRI_"))
chemin_pet = str(C.table("All_Subjects_Key_PET_"))

dx_spark = spark.read.csv(chemin_dx, header=True, inferSchema=False)
irm_spark = spark.read.csv(chemin_irm, header=True, inferSchema=False)
pet_spark = spark.read.csv(chemin_pet, header=True, inferSchema=False)

print("Lignes DXSUM   :", dx_spark.count())
print("Lignes Key_MRI :", irm_spark.count())
print("Lignes Key_PET :", pet_spark.count())

In [ ]:
# --- Le pipeline Spark : nettoyage + agregation + jointures ---
def pipeline_spark():
    # 1. Nettoyer les diagnostics.
    dx = (dx_spark.select("PTID", "EXAMDATE", "DIAGNOSIS")
          .withColumn("PTID", F.upper(F.trim(F.col("PTID"))))
          .withColumn("DIAGNOSIS", F.trim(F.col("DIAGNOSIS")))
          .withColumn("date_dx", F.to_date(F.trim(F.col("EXAMDATE")), "yyyy-MM-dd"))
          .filter(F.col("DIAGNOSIS").isin("1", "2", "3"))
          .filter(F.col("date_dx").isNotNull()))

    # 2. Ecarter les visites a codes contradictoires (meme regle que pandas).
    conflits = (dx.groupBy("PTID", "date_dx")
                  .agg(F.countDistinct("DIAGNOSIS").alias("n_codes"))
                  .filter(F.col("n_codes") > 1)
                  .select("PTID", "date_dx"))
    dx = dx.join(conflits, ["PTID", "date_dx"], "left_anti").dropDuplicates()

    # 3. Agreger par participant : nombre de dates et duree de suivi.
    resume = (dx.groupBy("PTID")
                .agg(F.countDistinct("date_dx").alias("n_dates"),
                     F.min("date_dx").alias("debut"),
                     F.max("date_dx").alias("fin"))
                .withColumn("suivi_jours", F.datediff("fin", "debut")))

    # 4. Appliquer les criteres longitudinaux.
    candidats = resume.filter((F.col("n_dates") >= 2) & (F.col("suivi_jours") >= 365))

    # 5. Joindre avec les catalogues d'imagerie (participants presents).
    sujets_irm = (irm_spark.select(F.upper(F.trim(F.col("subject_id"))).alias("PTID"))
                  .distinct())
    sujets_pet = (pet_spark.select(F.upper(F.trim(F.col("subject_id"))).alias("PTID"))
                  .distinct())

    avec_irm = candidats.join(sujets_irm, "PTID", "inner")
    avec_irm_pet = avec_irm.join(sujets_pet, "PTID", "inner")

    return {
        "visites_valides": dx.count(),
        "participants_suivis": resume.count(),
        "criteres_longitudinaux": candidats.count(),
        "+ present au catalogue IRM": avec_irm.count(),
        "+ present au catalogue PET": avec_irm_pet.count(),
    }

debut = time.perf_counter()
res_spark = pipeline_spark()
duree_spark = time.perf_counter() - debut

print("Resultats Spark :")
for cle, valeur in res_spark.items():
    print(f"  {cle:30s} : {valeur}")
print(f"\nTemps Spark : {duree_spark:.1f} s")

In [ ]:
# --- Le meme pipeline en pandas, pour verifier et comparer ---
def pipeline_pandas():
    dx = pd.read_csv(chemin_dx, dtype="string", encoding="utf-8-sig")
    dx["PTID"] = dx["PTID"].str.strip().str.upper()
    dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
    dx["date_dx"] = pd.to_datetime(dx["EXAMDATE"].str.strip(),
                                   format="%Y-%m-%d", errors="coerce")
    dx = dx.loc[dx["DIAGNOSIS"].isin(["1", "2", "3"]) & dx["date_dx"].notna(),
                ["PTID", "date_dx", "DIAGNOSIS"]]

    conf = dx.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
    conf = conf[conf > 1].index
    dx = dx.loc[~pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]]).isin(conf)]
    dx = dx.drop_duplicates()

    resume = dx.groupby("PTID").agg(n_dates=("date_dx", "nunique"),
                                    debut=("date_dx", "min"),
                                    fin=("date_dx", "max"))
    resume["suivi_jours"] = (resume["fin"] - resume["debut"]).dt.days
    candidats = resume.loc[(resume["n_dates"] >= 2) & (resume["suivi_jours"] >= 365)]

    irm = pd.read_csv(chemin_irm, dtype="string", usecols=["subject_id"])
    pet = pd.read_csv(chemin_pet, dtype="string", usecols=["subject_id"])
    s_irm = set(irm["subject_id"].str.strip().str.upper())
    s_pet = set(pet["subject_id"].str.strip().str.upper())

    avec_irm = candidats.index.intersection(s_irm)
    avec_irm_pet = avec_irm.intersection(s_pet)

    return {
        "visites_valides": len(dx),
        "participants_suivis": len(resume),
        "criteres_longitudinaux": len(candidats),
        "+ present au catalogue IRM": len(avec_irm),
        "+ present au catalogue PET": len(avec_irm_pet),
    }

debut = time.perf_counter()
res_pandas = pipeline_pandas()
duree_pandas = time.perf_counter() - debut

comparaison = pd.DataFrame({"Spark": res_spark, "pandas": res_pandas})
comparaison["identique"] = comparaison["Spark"] == comparaison["pandas"]
print("Verification : Spark donne-t-il les memes resultats que pandas ?")
display(comparaison)
print("\nTous identiques :", bool(comparaison["identique"].all()))
print(f"Temps pandas : {duree_pandas:.1f} s | Temps Spark : {duree_spark:.1f} s")

In [ ]:
# --- Mesure du passage a l'echelle : Spark sur 1, 2, puis tous les coeurs ---
# On relit la table en faisant varier le nombre de partitions pour illustrer
# le parallelisme (resultat a commenter dans la presentation).
mesures = []
for partitions in [1, 2, 4, 8]:
    spark.conf.set("spark.sql.shuffle.partitions", str(partitions))
    debut = time.perf_counter()
    _ = pipeline_spark()
    mesures.append({"partitions": partitions,
                    "temps_s": round(time.perf_counter() - debut, 2)})

mesures = pd.DataFrame(mesures)
display(mesures)

SORTIE = C.sortie("pipeline_spark")
comparaison.to_csv(SORTIE / "verification_spark_vs_pandas.csv", encoding="utf-8-sig")
mesures.to_csv(SORTIE / "temps_par_partitions.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{"temps_pandas_s": round(duree_pandas, 2),
               "temps_spark_s": round(duree_spark, 2)}]).to_csv(
    SORTIE / "temps_pandas_vs_spark.csv", index=False, encoding="utf-8-sig")
print("Resultats enregistres dans :", SORTIE)

spark.stop()